In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from datetime import datetime
import warnings
import os

warnings.filterwarnings('ignore')

# ==========================================
# 1. MySQL DB 연결
# ==========================================
db_user = 'root'
db_pw = 'jang1003'
db_host = 'localhost'
db_port = '3306'
db_name = 'quant_db'

db_connection_str = f'mysql+pymysql://{db_user}:{db_pw}@{db_host}:{db_port}/{db_name}'
engine = create_engine(db_connection_str)

def evaluate_portfolio_performance():
    # ==========================================
    # 2. DB에서 요약 로그(summary_log) 가져오기
    # ==========================================
    query = "SELECT * FROM portfolio_summary_log ORDER BY Last_Updated ASC"
    df = pd.read_sql(query, con=engine)


    # ==========================================
    # 3. 핵심 성과 지표 계산
    # ==========================================
    # 1) 주기별 수익률
    df['Return'] = df['Total_Portfolio_Value'].pct_change()

    # 2) 누적 수익률
    initial_value = df['Total_Portfolio_Value'].iloc[0]
    final_value = df['Total_Portfolio_Value'].iloc[-1]
    cumulative_return = ((final_value / initial_value) - 1) * 100

    # 3) MDD (최대 낙폭)
    df['Cumulative_Max'] = df['Total_Portfolio_Value'].cummax() #고점 판독기
    df['Drawdown'] = (df['Total_Portfolio_Value'] - df['Cumulative_Max']) / df['Cumulative_Max'] #모든 평가금액을 고점과 계산
    mdd = df['Drawdown'].min() * 100

    # 4) 샤프 지수 (연환산)
    risk_free_rate = 0.03 #현재 한국 은행들의 예금 금리 3%
    daily_rf_rate = risk_free_rate / 252 #연간 하루치 이율

    mean_return = df['Return'].mean() #내 포트폴리오 평균 수익률
    std_return = df['Return'].std() #표준편차(위험) 측정

    if std_return != 0:
        sharpe_ratio = ((mean_return - daily_rf_rate) / std_return) * np.sqrt(252)
    else:
        sharpe_ratio = 0.0

    # ==========================================
    # 4. 결과 CSV 추출 (하나의 파일에 누적 기록)
    # ==========================================
    # 세로형 표 대신, 한 줄(Row)로 들어가는 가로형 데이터로 만듭니다.
    metrics_data = {
        'Date': [datetime.now().strftime('%Y-%m-%d')],
        'Cumulative_Return_%': [round(cumulative_return, 2)],
        'MDD_%': [round(mdd, 2)],
        'Sharpe_Ratio': [round(sharpe_ratio, 2)]
    }
    metrics_df = pd.DataFrame(metrics_data)

    filename = "portfolio_MDDsharpe_history.csv"

    # 파일이 없으면 새로 만들고(헤더 포함), 이미 있으면 맨 밑에 내용만 추가(mode='a')
    if not os.path.exists(filename):
        metrics_df.to_csv(filename, index=False, mode='w')
    else:
        metrics_df.to_csv(filename, index=False, mode='a', header=False)


# 실행
evaluate_portfolio_performance()